# Graph Neural Network — Flood Forecasting (PyTorch Geometric)

Predicts streamflow 24 hours ahead for **all gauge sites simultaneously**,
while explicitly modelling the **river network topology** — water that falls
upstream arrives downstream later.

---

## What is a Graph Neural Network?

Standard neural networks expect a flat vector of features for each sample.
A **GNN** operates on a **graph**: a set of nodes (entities) connected by edges
(relationships). Instead of learning from each node in isolation, a GNN lets
each node *receive messages* from its neighbors and update its own
representation accordingly.

The core idea — **message passing** — runs in rounds:

```
Round 1: each node sees its immediate neighbors
Round 2: each node sees its neighbors' neighbors
...k rounds → each node has a view k hops away
```

Formally, one round of a **Graph Convolutional Network (GCN)** layer is:

```
H' = σ( D̂⁻½ Â D̂⁻½  H  W )
```

Where:
- `H`  — node feature matrix  (num_nodes × features)
- `Â`  — adjacency matrix with self-loops  (A + I)
- `D̂`  — degree matrix of Â  (for normalization)
- `W`  — learnable weight matrix
- `σ`  — non-linearity (ReLU)

In plain English: each node's new embedding is a **weighted average of
its own features and its neighbors' features**, passed through a linear layer.

---

## Why a GNN for River Networks?

The Missouri River Basin is a **directed acyclic graph**.  Water flows from
headwaters to the outlet, through tributaries, following a fixed topology.

```
   [Yellowstone R.]──┐
   [Milk R.]─────────┤
                     ▼
             [Missouri mainstem] ──► [Mississippi R.]
                     ▲
   [Platte R.]───────┤
   [Kansas R.]───────┘
```

Our previous models (LSTM, GRU) treated each site **independently** (series)
or concatenated all sites' features into one fat vector (parallel).  Neither
approach tells the model *which* sites are upstream of which.

A GNN can learn:
- Heavy rain at a headwater gauge → expect high flow 2 days later downstream
- A site's streamflow depends more on its upstream neighbor than a distant tributary
- Routing delay is proportional to distance / channel slope

---

## Architecture: Spatio-Temporal GNN (GCN + GRU)

We stack spatial (graph) and temporal (recurrent) processing:

```
Input per timestep t:   node_features  (num_nodes × F)
                              │
                     GCNConv (graph layer)
                              │
                     node_embeddings  (num_nodes × H)
                              │
Across window [t-W … t]:  sequence of embeddings
                              │
                     GRU (temporal layer) per node
                              │
                     final hidden state  (num_nodes × H)
                              │
                     Linear → prediction  (num_nodes × 1)
```

This is the same idea behind **DCRNN** (Diffusion Convolutional RNN),
which beat purely temporal models on traffic forecasting — a domain
structurally very similar to river routing.

## Inventory: What You Have vs. What You Need

| Ingredient | Status | Source |
|---|---|---|
| Streamflow observations (hourly) | ✅ Have | `flood-dataset-top30` W&B artifact |
| Weather forcing per site | ✅ Have | `flood-dataset-top30` W&B artifact |
| Site lat/lon | ✅ Have | Included in dataset |
| Static watershed attributes | ✅ Have | GAGES-II, HydroATLAS, NLDAS-2 (14 features) |
| **River network topology (edges)** | ⚠️ Need to build | NHDPlus or CAMELS |
| `torch_geometric` installed | ⚠️ Need to install | `uv add torch-geometric` |

### Data Source

Uses the **top-30 high flood-severity sites** dataset (`flood-dataset-top30`),
matching the LSTM notebooks for comparable results. This is hourly data with
14 dynamic features and 14 static watershed attributes.

### The Missing Piece: Graph Topology

A GNN needs an **edge list** — which gauge is upstream of which.
Two ways to get this:

1. **NHDPlus** (National Hydrography Dataset Plus) — the authoritative
   USGS river network.  Each USGS site has a `comid` that maps into the
   NHDPlus flowline network.  You can traverse `PlusFlow` table to find
   upstream/downstream neighbors.

2. **Proximity fallback** — connect each site to its K nearest neighbors
   by lat/lon.  Not hydrologically correct (ignores flow direction, divides)
   but lets you run experiments immediately.

This notebook implements both.  **Swap in NHDPlus edges when ready.**

In [2]:
"""
GNN Flood Forecasting Model (PyTorch Geometric + PyTorch)
Predicts streamflow 24 hours ahead for all sites simultaneously,
using the river network graph to share information between connected gauges.

Requires:
    uv add torch-geometric
    (installs torch_geometric, torch_scatter, torch_sparse)
"""
import torch
import torch.nn as nn
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

from torch_geometric.nn import GCNConv
from torch_geometric.data import Data, Batch

from src.preprocessing.preprocessing import processor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Same feature set as the LSTM notebooks so results are comparable.

STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg",
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

TARGET = "streamflow_cfs_mean"

config = {
    "input_cols": DYNAMIC_FEATURES + STATIC_FEATURES,
    "static_cols": STATIC_FEATURES,
    "target": "streamflow_cfs_target_24h",
    "train_split": 0.8,
    "val_split": 0.9,
    "file_path": "flood-dataset-top30",
    "file_name": "flood_model_top30",
    "table": "wandb.flood_model_top30",
    "lag_window": 1,
    "frequency": "hourly",
    "split_time_days": 30,
    "site_scaling": False,
}

# GNN-specific hyperparameters
GNN_CONFIG = {
    "window_size": 72,       # hours of history fed into the GRU (matches LSTM)
    "gcn_hidden": 32,        # embedding size output by each GCN layer
    "gru_hidden": 64,        # GRU hidden size
    "n_gcn_layers": 2,       # number of GCN rounds (= receptive field in hops)
    "dropout": 0.2,
    "k_neighbors": 3,        # K for the proximity-graph fallback
    "lr": 1e-3,
    "epochs": 50,
    "batch_size": 32,
    "patience": 8,
}

In [ ]:
# ── Load & preprocess data (reusing existing pipeline) ────────────────────────
pcr = processor(config)
pcr.pull_wandb()
print(pcr.df["site_id"].unique())
print(pcr.df.shape)

train_X, val_X, test_X, train_y, val_y, test_y = pcr.return_outputs()

In [ ]:
# ── Align sites so every split has identical timestamps across all nodes ──────
# GNNs require every node to be present at every timestep. We intersect
# observation_hour across sites to keep only shared timestamps.

def align_sites_by_date(
    X: pl.DataFrame, y: pl.DataFrame
) -> tuple[pl.DataFrame, pl.DataFrame]:
    sites = X["site_id"].unique().to_list()
    common_dates = None
    for site in sites:
        dates = set(X.filter(pl.col("site_id") == site)["observation_hour"].to_list())
        common_dates = dates if common_dates is None else common_dates & dates
    mask = X["observation_hour"].is_in(list(common_dates))
    print(f"  {len(common_dates)} common timestamps across {len(sites)} sites")
    return X.filter(mask), y.filter(mask)


print("Aligning train...")
train_X, train_y = align_sites_by_date(train_X, train_y)
print("Aligning val...")
val_X, val_y = align_sites_by_date(val_X, val_y)
print("Aligning test...")
test_X, test_y = align_sites_by_date(test_X, test_y)

## Step 1 — Build the Graph

This is the step that has no direct equivalent in the LSTM/GRU notebooks.

We need an **edge index**: a `(2, num_edges)` tensor where column `i` says
"node `edge_index[0, i]` is connected to node `edge_index[1, i]`".

### Option A — NHDPlus (correct, recommended)

```python
# TODO: fetch NHDPlus comids for each USGS site_id, then traverse
# the PlusFlow table to find upstream/downstream neighbors.
#
# pynhd (https://github.com/hyriver/pynhd) can query the NHDPlus REST API:
#
#   from pynhd import NLDI
#   nldi = NLDI()
#   # Get the upstream basin for site 06934000
#   basin = nldi.get_basins("usgs-sw", ["06934000"])
#
# Once you have the comid → comid adjacency, map back to site_id indices.
```

### Option B — K-Nearest Neighbors by lat/lon (placeholder)

We use this below.  It connects each gauge to its 3 nearest gauges by
geographic distance.  It ignores flow direction but is runnable today.

In [ ]:
# ── Build graph edges ─────────────────────────────────────────────────────────

# latitude/longitude are in the feature columns; extract one row per site.
# The preprocessor keeps them as scaled values, so we pull from the raw df.
site_meta = (
    pcr.df
    .select(["site_id", "latitude", "longitude"])
    .unique("site_id")
    .sort("site_id")
)
sites_ordered = site_meta["site_id"].to_list()
site_to_idx = {s: i for i, s in enumerate(sites_ordered)}
num_nodes = len(sites_ordered)

coords = site_meta.select(["latitude", "longitude"]).to_numpy()
print("Nodes:", num_nodes)
print("Site order:", sites_ordered)

# ── Option A placeholder ──────────────────────────────────────────────────────
# When NHDPlus edges are available, replace `build_knn_edges` with a function
# that reads the comid adjacency and maps to site indices.
#
# nhd_edges = build_nhd_edges(sites_ordered)  # returns (2, E) tensor

# ── Option B: KNN proximity graph ────────────────────────────────────────────
def build_knn_edges(coords: np.ndarray, k: int) -> torch.Tensor:
    """Connect each node to its k nearest geographic neighbors (undirected)."""
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="haversine").fit(
        np.deg2rad(coords)
    )
    _, indices = nbrs.kneighbors(np.deg2rad(coords))

    src, dst = [], []
    for i, neighbors in enumerate(indices):
        for j in neighbors[1:]:
            src.append(i)
            dst.append(j)
            src.append(j)
            dst.append(i)

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_index = torch.unique(edge_index, dim=1)
    return edge_index


edge_index = build_knn_edges(coords, k=GNN_CONFIG["k_neighbors"])
print(f"Edges: {edge_index.shape[1]} (undirected, k={GNN_CONFIG['k_neighbors']})")

fig, ax = plt.subplots(figsize=(8, 6))
lons = coords[:, 1]
lats = coords[:, 0]
for e in range(edge_index.shape[1]):
    u, v = edge_index[0, e].item(), edge_index[1, e].item()
    ax.plot([lons[u], lons[v]], [lats[u], lats[v]], "b-", alpha=0.3, lw=1)
ax.scatter(lons, lats, c="red", zorder=5, s=80)
for i, site in enumerate(sites_ordered):
    ax.annotate(site, (lons[i], lats[i]), fontsize=7, ha="right")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Gauge Graph — KNN (k={GNN_CONFIG['k_neighbors']}) proximity edges")
ax.set_title("Replace with NHDPlus river-network edges for real hydrology",
             fontsize=9, color="orange", loc="left")
plt.tight_layout()
plt.show()

## Step 2 — Build Temporal Graph Sequences

The LSTM notebook used shape `(samples, window, features)` for a *single* site.

For the GNN we need a different shape — one that preserves the node (site)
dimension so the graph convolution can operate across sites at each timestep:

```
X_seq:  (num_windows, window_size, num_nodes, node_features)
              │             │           │           │
          batches      time steps     sites    features per site
y_seq:  (num_windows, num_nodes)   ← one target per site per window
```

With hourly data and a 72-hour window, each sample captures 3 days of
spatial-temporal context.

The GCN processes `data.x` spatially, then the GRU processes the
embedded sequence temporally.

In [ ]:
# ── Build (windows, nodes, time, features) tensors ───────────────────────────

def build_graph_sequences(
    X: pl.DataFrame,
    y: pl.DataFrame,
    site_to_idx: dict,
    window_size: int,
    drop_cols: list[str] | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Build sliding-window sequences with a node (site) dimension for the GNN.

    Returns:
        X_seq: (num_windows, window_size, num_nodes, num_features)  float32
        y_seq: (num_windows, num_nodes)                             float32
    """
    if drop_cols is None:
        drop_cols = ["site_id", "observation_hour"]

    sites_ordered = sorted(site_to_idx, key=site_to_idx.get)

    site_X, site_y = {}, {}
    for site in sites_ordered:
        mask = X["site_id"] == site
        site_X[site] = torch.tensor(
            X.filter(mask).drop(drop_cols).to_numpy(), dtype=torch.float32
        )
        site_y[site] = torch.tensor(
            y.filter(mask).to_numpy().flatten(), dtype=torch.float32
        )

    X_stacked = torch.stack([site_X[s] for s in sites_ordered], dim=0)  # (N, T, F)
    y_stacked = torch.stack([site_y[s] for s in sites_ordered], dim=0)  # (N, T)

    T = X_stacked.shape[1]
    num_windows = T - window_size
    if num_windows <= 0:
        raise ValueError(f"Not enough timesteps ({T}) for window_size={window_size}")

    X_windows = torch.stack(
        [X_stacked[:, t : t + window_size, :] for t in range(num_windows)],
        dim=0,
    ).permute(0, 2, 1, 3)  # (W, window_size, N, F)

    y_windows = y_stacked[:, window_size:].T  # (W, N)

    nan_x = X_windows.isnan().any(dim=(1, 2, 3))
    nan_y = y_windows.isnan().any(dim=1)
    keep = ~(nan_x | nan_y)
    print(f"  Dropped {(~keep).sum().item()} NaN windows, kept {keep.sum().item()}")
    return X_windows[keep], y_windows[keep]


WINDOW_SIZE = GNN_CONFIG["window_size"]

print("Building train sequences...")
X_train, y_train = build_graph_sequences(train_X, train_y, site_to_idx, WINDOW_SIZE)
print("Building val sequences...")
X_val, y_val = build_graph_sequences(val_X, val_y, site_to_idx, WINDOW_SIZE)
print("Building test sequences...")
X_test, y_test = build_graph_sequences(test_X, test_y, site_to_idx, WINDOW_SIZE)

print(f"\nX_train: {X_train.shape}  (windows, time, nodes, features)")
print(f"y_train: {y_train.shape}  (windows, nodes)")

## Step 3 — Define the GNN Model

Our architecture: **GCN-GRU** (also called ST-GNN or Graph-WaveNet lite).

```
For each timestep t in [t-W, ..., t]:
    node_features (N × F)  →  GCNConv  →  node_embeddings (N × H_gcn)

GRU sees a sequence of length W, each step being (N × H_gcn):
    → final hidden state (N × H_gru)

Linear head:
    (N × H_gru)  →  (N × 1)  → predicted streamflow per site
```

Key difference from a vanilla GRU:
- The *input at each timestep* is not a flat vector but a **graph-convolved
  embedding** that already blends information from neighboring sites.
- The GRU then models *how that spatially-enriched signal evolves over time*.

> **Note on GCN layers (`n_gcn_layers`)**  
> With 1 GCN layer, each node sees 1-hop neighbors.  
> With 2 layers, each node sees 2-hop neighbors (neighbors of neighbors).  
> More layers = wider spatial context, but also more over-smoothing risk.

In [ ]:
# ── Model definition ──────────────────────────────────────────────────────────

class GCNGRU(nn.Module):
    """
    Spatio-Temporal GNN: GCN for spatial aggregation, GRU for temporal modelling.

    Args:
        in_features:  node feature dimension (F)
        gcn_hidden:   output dimension of each GCN layer
        gru_hidden:   GRU hidden state size
        n_gcn_layers: number of GCN message-passing rounds per timestep
        dropout:      dropout probability
    """

    def __init__(
        self,
        in_features: int,
        gcn_hidden: int,
        gru_hidden: int,
        n_gcn_layers: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()

        # Stack of GCN layers (applied identically at each timestep)
        gcn_dims = [in_features] + [gcn_hidden] * n_gcn_layers
        self.gcn_layers = nn.ModuleList(
            [GCNConv(gcn_dims[i], gcn_dims[i + 1]) for i in range(n_gcn_layers)]
        )
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

        # GRU processes the sequence of GCN-embedded timesteps
        # input_size = gcn_hidden (one embedding per node per step)
        self.gru = nn.GRU(
            input_size=gcn_hidden,
            hidden_size=gru_hidden,
            batch_first=True,  # (batch, seq, features)
        )

        # Per-node output head
        self.head = nn.Linear(gru_hidden, 1)

    def forward(
        self,
        x: torch.Tensor,        # (batch, window_size, num_nodes, F)
        edge_index: torch.Tensor,  # (2, E)  — shared across all batches/timesteps
    ) -> torch.Tensor:           # (batch, num_nodes)

        batch_size, window_size, num_nodes, _ = x.shape

        # 1. Apply GCN to every (batch, timestep) slice
        #    Reshape to (batch * window_size * num_nodes, F) for GCNConv
        gcn_out_list = []
        for t in range(window_size):
            # x_t: (batch, num_nodes, F)
            x_t = x[:, t, :, :]  
            # Flatten batch dimension: treat each (batch, graph) independently
            # GCNConv doesn't natively support batched graphs — we loop over batch
            # (for production, use torch_geometric Batch for efficiency)
            h_t_list = []
            for b in range(batch_size):
                h = x_t[b]  # (num_nodes, F)
                for gcn in self.gcn_layers:
                    h = self.relu(gcn(h, edge_index))
                    h = self.dropout(h)
                h_t_list.append(h)  # (num_nodes, gcn_hidden)
            gcn_out_list.append(torch.stack(h_t_list, dim=0))  # (batch, N, gcn_hidden)

        # gcn_out: (batch, window_size, num_nodes, gcn_hidden)
        gcn_out = torch.stack(gcn_out_list, dim=1)

        # 2. Apply GRU per node across the time window
        #    Reshape to (batch * num_nodes, window_size, gcn_hidden)
        gcn_out = gcn_out.permute(0, 2, 1, 3)  # (batch, N, window, gcn_hidden)
        gcn_flat = gcn_out.reshape(batch_size * num_nodes, window_size, -1)

        _, h_n = self.gru(gcn_flat)  # h_n: (1, batch*N, gru_hidden)
        h_n = h_n.squeeze(0)                 # (batch*N, gru_hidden)

        # 3. Output head
        out = self.head(h_n)                 # (batch*N, 1)
        out = out.reshape(batch_size, num_nodes)  # (batch, N)
        return out


# Instantiate
in_features = X_train.shape[-1]  # node feature dimension
model = GCNGRU(
    in_features=in_features,
    gcn_hidden=GNN_CONFIG["gcn_hidden"],
    gru_hidden=GNN_CONFIG["gru_hidden"],
    n_gcn_layers=GNN_CONFIG["n_gcn_layers"],
    dropout=GNN_CONFIG["dropout"],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

In [ ]:
# ── Quick sanity-check forward pass ──────────────────────────────────────────
sample_batch = X_train[:4].to(DEVICE)
edge_index_dev = edge_index.to(DEVICE)

with torch.no_grad():
    sample_out = model(sample_batch, edge_index_dev)

print(f"Input shape:  {sample_batch.shape}")
print(f"Output shape: {sample_out.shape}   <- (batch, num_nodes)")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────

UNDER_PREDICT_PENALTY = 2.0

def asymmetric_mse(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """Penalise under-predictions more — safer for flood forecasting."""
    error = y_true - y_pred
    weight = torch.where(error > 0, UNDER_PREDICT_PENALTY, 1.0)
    return (weight * error ** 2).mean()


optimizer = torch.optim.Adam(model.parameters(), lr=GNN_CONFIG["lr"])

X_train_dev = X_train.to(DEVICE)
y_train_dev = y_train.to(DEVICE)
X_val_dev   = X_val.to(DEVICE)
y_val_dev   = y_val.to(DEVICE)

best_val_loss = float("inf")
patience_counter = 0
train_losses, val_losses = [], []

BATCH = GNN_CONFIG["batch_size"]
n_train = X_train_dev.shape[0]

for epoch in range(GNN_CONFIG["epochs"]):
    # ── Train ──
    model.train()
    epoch_loss = 0.0
    perm = torch.randperm(n_train)
    for start in range(0, n_train, BATCH):
        idx = perm[start : start + BATCH]
        xb = X_train_dev[idx]
        yb = y_train_dev[idx]

        optimizer.zero_grad()
        pred = model(xb, edge_index_dev)
        loss = asymmetric_mse(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)

    train_loss = epoch_loss / n_train

    # ── Validate ──
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_dev, edge_index_dev)
        val_loss = asymmetric_mse(val_pred, y_val_dev).item()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | train={train_loss:.4f} | val={val_loss:.4f}")

    # ── Early stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= GNN_CONFIG["patience"]:
            print(f"Early stop at epoch {epoch+1}")
            break

model.load_state_dict(best_weights)
print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# ── Loss curves ───────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Val")
plt.xlabel("Epoch")
plt.ylabel("Asymmetric MSE (scaled)")
plt.title("GCN-GRU Training Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    pred_test_scaled = model(X_test.to(DEVICE), edge_index_dev).cpu()

# Inverse-transform to real CFS (target_scaler expects 2D input)
pred_flat = pred_test_scaled.reshape(-1, 1).to(torch.float64)
actual_flat = y_test.reshape(-1, 1).to(torch.float64)

pred_real = pcr.target_scaler.inverse_transform(pred_flat).numpy().reshape(pred_test_scaled.shape)
actual_real = pcr.target_scaler.inverse_transform(actual_flat).numpy().reshape(y_test.shape)

print(f"{'Site':<12} {'Actual Mean':>14} {'Predicted Mean':>16} {'MAE':>12}")
print("-" * 58)
for i, site in enumerate(sites_ordered):
    mae = np.abs(actual_real[:, i] - pred_real[:, i]).mean()
    print(
        f"{site:<12} {actual_real[:, i].mean():>12.1f} CFS "
        f"{pred_real[:, i].mean():>12.1f} CFS  {mae:>8.1f} CFS"
    )

In [ ]:
# ── Time-series plot: one site ────────────────────────────────────────────────
site_to_plot = sites_ordered[0]
idx = site_to_idx[site_to_plot]

n_show = min(500, len(actual_real))
plt.figure(figsize=(14, 5))
plt.plot(actual_real[:n_show, idx], label="Actual", alpha=0.8)
plt.plot(pred_real[:n_show, idx], label="Predicted (GCN-GRU)", alpha=0.8)
plt.xlabel("Time step (hours)")
plt.ylabel("Streamflow (CFS)")
plt.title(f"GCN-GRU Predictions vs Actual — Site {site_to_plot} (first {n_show} test hours)")
plt.legend()
plt.tight_layout()
plt.show()

## What's Next

### Immediate improvements (no new data needed)

| Idea | Why |
|---|---|
| **Directed edges** — point edges downstream only | Water doesn't flow uphill; directionality helps |
| **Edge features** — distance, elevation drop | Let the model learn routing delay from the edge itself |
| **Static node features** — GAGES-II / HydroATLAS | Soil type, slope, imperviousness all affect runoff |
| **Batched graph** — use `torch_geometric.data.Batch` | Removes the Python loop over batch, 10-100× faster |
| **`GraphSAGE`** instead of `GCNConv` | Inductive: works on unseen sites at inference time |

### Requires new data

| Idea | Source |
|---|---|
| **NHDPlus river topology** | `pynhd` → query by USGS site_id comid |
| **Routing delay as edge weight** | NHDPlus `LengthKM` + channel velocity estimates |
| **Upstream contributing area** | NHDPlus `TotDASqKM` — strong predictor of peak flow |

### Architecture upgrades

| Model | Paper | Key idea |
|---|---|---|
| **DCRNN** | Li et al. 2018 | Graph diffusion instead of simple convolution; handles directed graphs natively |
| **Graph WaveNet** | Wu et al. 2019 | Adaptive adjacency matrix — learns graph structure from data |
| **AGCRN** | Bai et al. 2020 | Node-specific GRU weights; strong on heterogeneous graphs |

All three have open PyTorch implementations and have been applied directly
to streamflow or flood forecasting tasks in the literature.